# LaLonde-Dehejia-Wahba DS-WGAN (PSID)
**Athey, Imbens, Metzger & Munro (2020)**

Two-stage conditional WGAN on the real LaLonde-Dehejia-Wahba PSID dataset using `wgan_jax.py`:

- **Stage 1**: Learn $X \mid W$ (8 covariates conditional on treatment)
- **Stage 2**: Learn $Y(=\text{re78}) \mid X, W$ (earnings outcome conditional on covariates and treatment)
- **Generation**: Sample $\tilde{X}$ from Stage 1, then $\tilde{Y}$ from Stage 2 conditioned on $(\tilde{X}, W)$

PSID sample: 2,675 observations (185 treated from the experiment + 2,490 PSID controls).

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wgan_jax import WGAN, compare_dfs

## 1. Load & Explore PSID LaLonde Data

In [ ]:
# Variable classifications (following the paper, Section 3.2)
CONTINUOUS_VARS = ["age", "education", "re74", "re75"]
CATEGORICAL_VARS = ["black", "hispanic", "married", "nodegree", "u74", "u75"]
OUTCOME_VAR = "re78"
TREATMENT_VAR = "t"
COVARIATE_COLS = CONTINUOUS_VARS + CATEGORICAL_VARS
COND_STAGE2 = COVARIATE_COLS + [TREATMENT_VAR]

# Bounds
CONTINUOUS_LOWER_BOUNDS = {"age": 17, "education": 0, "re74": 0, "re75": 0}
OUTCOME_LOWER_BOUNDS = {"re78": 0}

# Load PSID data
import pyarrow.feather as pf
df = pf.read_table("../data/psid.feather").to_pandas()
# u74/u75 are int32 in the feather file; cast to float for consistency
df["u74"] = df["u74"].astype(float)
df["u75"] = df["u75"].astype(float)

print(f"N = {len(df)}, treated = {int(df['t'].sum())}, control = {int((1 - df['t']).sum())}")
df.describe()

In [ ]:
# Covariate imbalance by treatment group (compare with paper Table 1, PSID column)
print("=== Means by treatment group ===")
print(df.groupby("t").mean().round(2).T)
print()
print("=== Std by treatment group ===")
print(df.groupby("t").std().round(2).T)

In [ ]:
# Earnings distributions show strong zero-inflation
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, var in zip(axes, ["re74", "re75", "re78"]):
    for t_val, label, color in [(1, "treated", "C0"), (0, "control", "C1")]:
        ax.hist(df.loc[df["t"] == t_val, var], bins=30, alpha=0.5,
                label=label, color=color, density=True)
    ax.set_title(var)
    ax.legend(prop={"size": 8})
fig.suptitle("Earnings distributions by treatment group")
fig.tight_layout()
plt.show()

## 2. Stage 1: X | W

Train a conditional WGAN to generate the 8 pre-treatment covariates conditional on treatment $W$.
Architecture follows the paper: 3 hidden layers of 128 units, batch size 512 (PSID setting).

In [ ]:
wgan_stage1 = WGAN(
    df=df,
    continuous_vars=CONTINUOUS_VARS,
    categorical_vars=CATEGORICAL_VARS,
    conditioning_vars=[TREATMENT_VAR],
    continuous_lower_bounds=CONTINUOUS_LOWER_BOUNDS,
    critic_hidden=[128, 128, 128],
    generator_hidden=[128, 128, 128],
    critic_steps=15,
    gp_factor=5.0,
    seed=0,
)

In [ ]:
t0 = time.time()
history1 = wgan_stage1.train(max_epochs=1000, batch_size=512, print_every=100)
print(f"\nStage 1 training time: {time.time() - t0:.1f}s")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(pd.Series(history1["wd_train"]).rolling(50).mean(), label="WD train")
plt.plot(pd.Series(history1["wd_test"]).rolling(50).mean(), label="WD test")
plt.xlabel("Epoch"); plt.ylabel("Wasserstein Distance")
plt.title("Stage 1: X | W (LaLonde PSID)")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
df_stage1 = wgan_stage1.generate(df[[TREATMENT_VAR]], seed=0)

compare_dfs(
    df[COVARIATE_COLS + [TREATMENT_VAR]],
    df_stage1,
    histogram=dict(variables=["age", "education", "re74", "re75"], nrow=2, ncol=2),
    table_groupby=[TREATMENT_VAR],
)

## 3. Stage 2: re78 | X, W

Train a second WGAN to generate 1978 earnings conditional on all 8 covariates and treatment.

In [ ]:
wgan_stage2 = WGAN(
    df=df,
    continuous_vars=[OUTCOME_VAR],
    categorical_vars=[],
    conditioning_vars=COND_STAGE2,
    continuous_lower_bounds=OUTCOME_LOWER_BOUNDS,
    critic_hidden=[128, 128, 128],
    generator_hidden=[128, 128, 128],
    critic_steps=15,
    gp_factor=5.0,
    seed=1,
)

In [ ]:
t0 = time.time()
history2 = wgan_stage2.train(max_epochs=1000, batch_size=512, print_every=100)
print(f"\nStage 2 training time: {time.time() - t0:.1f}s")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(pd.Series(history2["wd_train"]).rolling(50).mean(), label="WD train")
plt.plot(pd.Series(history2["wd_test"]).rolling(50).mean(), label="WD test")
plt.xlabel("Epoch"); plt.ylabel("Wasserstein Distance")
plt.title("Stage 2: re78 | X, W (LaLonde PSID)")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
df_stage2 = wgan_stage2.generate(df[COND_STAGE2], seed=0)

compare_dfs(
    df[[OUTCOME_VAR, TREATMENT_VAR]],
    df_stage2[[OUTCOME_VAR, TREATMENT_VAR]],
    histogram=dict(variables=[OUTCOME_VAR], nrow=1, ncol=1),
    table_groupby=[TREATMENT_VAR],
)

## 4. Full DS-WGAN Generation Pipeline

Chain both stages: sample $\tilde{X}$ from Stage 1, then $\tilde{Y}$ from Stage 2 conditioned on $(\tilde{X}, W)$.
Compare marginal distributions, correlations, and conditional means against the real data (paper Figures 1-3).

In [ ]:
def generate_ds_wgan(wgan_s1, wgan_s2, df_w, seed=0):
    """Two-stage DS-WGAN generation.

    1. Generate X_tilde | W from Stage 1
    2. Generate Y_tilde | X_tilde, W from Stage 2
    """
    df_x = wgan_s1.generate(df_w, seed=seed)
    df_y = wgan_s2.generate(df_x[COND_STAGE2], seed=seed + 1)
    return df_y

In [ ]:
df_syn = generate_ds_wgan(wgan_stage1, wgan_stage2, df[[TREATMENT_VAR]], seed=42)

compare_dfs(
    df, df_syn,
    histogram=dict(
        variables=["re78", "age", "education", "re74", "re75"],
        nrow=2, ncol=3,
    ),
    scatterplot=dict(x=["age", "re74"], y=["re78"], samples=500, smooth=0.5),
    table_groupby=[TREATMENT_VAR],
)

### Conditional histograms: re78 | re74 = 0 vs re74 > 0

The paper (Figure 3) highlights that the conditional distribution of 1978 earnings looks very different
depending on whether 1974 earnings are zero or positive. A good generator should capture this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, (cond_label, cond_fn) in zip(axes, [
    ("re74 = 0", lambda d: d[d["re74"] == 0]),
    ("re74 > 0", lambda d: d[d["re74"] > 0]),
]):
    real_sub = cond_fn(df)
    fake_sub = cond_fn(df_syn)
    ax.hist([real_sub["re78"], fake_sub["re78"]], bins=30, density=True,
            histtype="bar", label=["real", "fake"], color=["blue", "red"], alpha=0.6)
    ax.set_title(f"re78 | {cond_label}")
    ax.legend(prop={"size": 8})

fig.suptitle("Conditional distributions (cf. paper Figure 3)")
fig.tight_layout()
plt.show()

## 5. Population ATT from the Generator

For treated units, hold Stage 1 covariates $\tilde{X}$ fixed and re-run Stage 2 with $W$ flipped to 0
to get counterfactual $\tilde{Y}(0)$. The population ATT is
$\tau = \mathbb{E}[\tilde{Y}(1) - \tilde{Y}(0) \mid W=1]$.

In [ ]:
N_pop = 1_000_000
w_pop = np.random.choice(df[TREATMENT_VAR].values, N_pop, replace=True)
df_pop = generate_ds_wgan(
    wgan_stage1, wgan_stage2, pd.DataFrame({TREATMENT_VAR: w_pop}), seed=0
)

# For treated units: Y(1) is already generated. Get Y(0) by flipping W.
treated_mask = df_pop[TREATMENT_VAR] == 1
df_treated = df_pop[treated_mask].copy()

y1 = df_treated[OUTCOME_VAR].values

# Counterfactual: same covariates, W=0
df_cf_cond = df_treated[COND_STAGE2].copy()
df_cf_cond[TREATMENT_VAR] = 0.0
df_cf = wgan_stage2.generate(df_cf_cond, seed=99)
y0 = df_cf[OUTCOME_VAR].values

pop_att = y1.mean() - y0.mean()
print(f"Generator population ATT: ${pop_att:,.0f}")
print(f"Experimental benchmark ATT (Dehejia-Wahba): ~$1,794")

## 6. Save Trained Models

In [ ]:
wgan_stage1.save("../models/lalonde_psid_stage1")
wgan_stage2.save("../models/lalonde_psid_stage2")